In [1]:
from lmetk.data.datasets import Dataset_ETT_hour


%reload_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
data = Dataset_ETT_hour(data_dir='datasets/ETT-small')

#/home/levtel/projects/TabPHM/datasets/ETT-small/ETTh1.csv

In [3]:
import hydra
datasets = ['ETTh1']

for data_name in datasets:
#data_name = 'ETTh1'
    for flag in ['train', 'val', 'test']:
        with hydra.initialize(
                version_base="1.3",
                config_path='configs',
                job_name="run"
            ):
            #print('Current config file: ', config_file)
            parameters = hydra.compose(
                config_name="run.yaml",
                overrides=[f'dataset={data_name}', f'dataset.loader.flag={flag}'], 
                return_hydra_config=True, 
            )
            dataset_loader1 = hydra.utils.instantiate(parameters.dataset.loader)
            
    
            print(f"{flag} dataset size", dataset_loader1.__len__())

train dataset size 8161
val dataset size 2785
test dataset size 2785


### Behaviour of not refactored dataset

In [4]:
with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset=ETTh1', f'dataset.loader.flag=val'], 
        return_hydra_config=True, 
    )
    dataset_loader1 = hydra.utils.instantiate(parameters.dataset.loader)

for i in range(2783, 2787):
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_loader1.__getitem__(i)
    print('Index', i)
    print(f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}")



Index 2783
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2784
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2785
seq_x shape: (384, 1), seq_y shape: (191, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (191, 4)
Index 2786
seq_x shape: (384, 1), seq_y shape: (190, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (190, 4)


### Refactored version

In [5]:

from lmetk.data.datasets import TimeSeriesDataset

data_name = 'ETTh1_refactored'

with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset={data_name}'], 
        return_hydra_config=True, 
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)


# Get params of splitting and sliding
data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = TimeSeriesDataset(data=splitted_data['train'], 
                  time_features=splitted_timestamps['train'], 
                  seq_len=seq_len, 
                  label_len=label_len, 
                  pred_len=pred_len)

dataset_val = TimeSeriesDataset(data=splitted_data['val'],
                    time_features=splitted_timestamps['val'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
dataset_test = TimeSeriesDataset(data=splitted_data['test'],
                    time_features=splitted_timestamps['test'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

Train dataset size: 8161
Validation dataset size: 2785
Test dataset size: 5805


In [6]:
for i in range(2783, 2787):
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(i)
    print('Index', i)
    print(f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}")




Index 2783
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2784
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)


AssertionError: Target sequence indices out of bounds

In [27]:
# for i in range(2875, 2885):
#     seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(i)
#     print('Index', i)
#     print(f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}")

